In [1]:
import pandas as pd

# ============================================================
# 데이터 불러오기
# ============================================================

from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

train_full = pd.read_parquet(PROJECT_ROOT / 'data' / '2_train_full_fe.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'data' / '2_test_fe.parquet')

In [2]:
# ============================================================
# 컬럼군 정의
# ============================================================

# 종속변수군
target_col = 'target_maxrise_6h'

# 독립변수군
feature_cols = [
    col for col in train_full.columns
    if col not in ('timestamp', 'event_id', target_col)
]

assert len(feature_cols) == 32, (
    f'독립변수 개수 이상: 예상 32개, 현재 {len(feature_cols)}개'
)

## 4. Tidal, Normal 데이터 분할

In [3]:
# ============================================================
# 4-1. Tidal 데이터 필터링
# ============================================================

# 조건 설정
tidal_cond = (
    (train_full['wl_diff_6h'] < -1) &  # 수위하강이 발생하면서(0이 아닌)
    ((train_full[target_col] + train_full['wl_diff_6h']).abs() < 2)  # 과거와 미래 6시간 수위변동이 반대: 주기성의 일부
)

# 원본 station_id 결측치 검증
assert not train_full['station_id'].isnull().any(), "station_id 결측치 존재"

tidal_station_df = pd.DataFrame({
    'total_count' : train_full['station_id'].value_counts(),
    'tidal_count' : train_full.loc[tidal_cond, 'station_id'].value_counts()
}).fillna(0).astype(int)

# 한번 더 검증
assert not tidal_station_df.isnull().any().any(), "아직 NaN 있음"

# 관측소별 tidal 비율 추가
tidal_station_df['tidal_ratio(%)'] = (
    tidal_station_df['tidal_count'] / tidal_station_df['total_count'] * 100
)

# tidal_count기준 내림차순 정렬
tidal_station_df = tidal_station_df.sort_values(
    'tidal_count', ascending=False
)

tidal_stations = (
    tidal_station_df
    .head(9)  # Train(Validation 제외)에서 Tidal 발생 비율 10% 초과였던 9개 관측소 유지
    .index
)

print(tidal_stations)

Index(['ST0326', 'ST0358', 'ST0292', 'ST0347', 'ST0159', 'ST0110', 'ST0415',
       'ST0313', 'ST0234'],
      dtype='object', name='station_id')


In [4]:
# ============================================================
# 4-2. Tidal / Normal 데이터 분리
# ============================================================

# 4-1. Train 분리
tidal_train = train_full.loc[
    train_full['station_id'].isin(tidal_stations)
].copy()

normal_train = train_full.loc[
    ~train_full['station_id'].isin(tidal_stations)
].copy()

# 4-2. Test 분리
tidal_test = test.loc[
    test['station_id'].isin(tidal_stations)
].copy()

normal_test = test.loc[
    ~test['station_id'].isin(tidal_stations)
].copy()

# 4-3. 분리 결과 확인
print('<Train 분리 결과>')
print(f'Train 전체 : {len(train_full):,}')
print(f'Tidal      : {len(tidal_train):,}')
print(f'Normal     : {len(normal_train):,}')

print('\n<Test 분리 결과>')
print(f'Test 전체  : {len(test):,}')
print(f'Tidal      : {len(tidal_test):,}')
print(f'Normal     : {len(normal_test):,}')

# 4-4. 데이터 분리 검증
assert len(tidal_train) + len(normal_train) == len(train_full)
assert len(tidal_test) + len(normal_test) == len(test)

print('\n데이터 분리 검증 완료')

<Train 분리 결과>
Train 전체 : 749,609
Tidal      : 14,251
Normal     : 735,358

<Test 분리 결과>
Test 전체  : 186,263
Tidal      : 3,573
Normal     : 182,690

데이터 분리 검증 완료


## 5. 독립변수 / 종속변수 설정

In [5]:
# ============================================================
# 5. 독립변수 / 종속변수 분리
# ============================================================

assert len(feature_cols) == 32, (
    f'독립변수 개수 이상: 예상 32개, 현재 {len(feature_cols)}개'
)

# Tidal
X_tidal_train = tidal_train[feature_cols].copy()
X_tidal_test = tidal_test[feature_cols].copy()

y_tidal_train = tidal_train[target_col]

# Normal
X_normal_train = normal_train[feature_cols].copy()
X_normal_test = normal_test[feature_cols].copy()

y_normal_train = normal_train[target_col]

# 데이터 분리 검증
assert X_tidal_train.index.equals(y_tidal_train.index)
assert X_normal_train.index.equals(y_normal_train.index)

assert X_tidal_train.columns.equals(X_tidal_test.columns)
assert X_normal_train.columns.equals(X_normal_test.columns)

print('독립변수 / 종속변수 분리 검증 완료')

독립변수 / 종속변수 분리 검증 완료


## 6. Tidal 데이터셋 모델 생성: 단일 홀드아웃, CatBoost, ExtraTree 앙상블

In [6]:
# ============================================================
# 6-1. CatBoost에 넣을 범주형 변수 처리
# ============================================================

for X in [
    X_tidal_train, X_tidal_test,
    X_normal_train, X_normal_test
]:
    X['station_id'] = X['station_id'].astype(str)

In [7]:
# ============================================================
# 6-2. CatBoost 모델로 tidal 예측
# ============================================================
from catboost import CatBoostRegressor

# Tidal CatBoost 모델

cat_tidal = CatBoostRegressor()

cat_tidal.load_model(
    str(PROJECT_ROOT / 'model' / 'cat_tidal.cbm')
)

pred_cat_tidal = cat_tidal.predict(X_tidal_test)


In [8]:
# ============================================================
# 6-3. ExtraTrees 모델에 넣을 범주형 변수 처리: OneHotEncoding
# ============================================================

import joblib

from sklearn.preprocessing import OneHotEncoder

# OneHotEncoder 정의 및 학습
ohe = joblib.load(
    str(PROJECT_ROOT / 'model' / 'ohe_tidal_station_id.joblib')
)

# Train / Test 원핫인코딩
X_tidal_train_ohe = pd.concat([
    X_tidal_train.drop(columns='station_id'),  # 기존에 있던 'station_id' 컬럼은 제거
    pd.DataFrame(  # 원핫인코딩 결과 데이터프레임으로 만들어서 붙이기
        ohe.transform(X_tidal_train[['station_id']]),  
        columns=ohe.get_feature_names_out(),  # 컬럼 이름을 해당 station_id로 받아오기
        index=X_tidal_train.index
    )
], axis=1)

X_tidal_test_ohe = pd.concat([
    X_tidal_test.drop(columns='station_id'),
    pd.DataFrame(
        ohe.transform(X_tidal_test[['station_id']]),
        columns=ohe.get_feature_names_out(),
        index=X_tidal_test.index
    )
], axis=1)

# 검증
assert 'station_id' not in X_tidal_train_ohe.columns
assert X_tidal_train_ohe.columns.equals(X_tidal_test_ohe.columns)
assert X_tidal_train_ohe.index.equals(y_tidal_train.index)

# 원핫인코딩 전후 Shape 비교
print(f'Tidal Train : {X_tidal_train.shape} -> {X_tidal_train_ohe.shape}')
print(f'Tidal Test  : {X_tidal_test.shape} -> {X_tidal_test_ohe.shape}')

Tidal Train : (14251, 32) -> (14251, 40)
Tidal Test  : (3573, 32) -> (3573, 40)


In [9]:
# ============================================================
# 6-4. ExtraTrees 모델로 tidal 예측
# ============================================================
from sklearn.ensemble import ExtraTreesRegressor

# Tidal ExtraTrees 모델
et_tidal = joblib.load(
    str(PROJECT_ROOT / 'model' / 'et_tidal_extratrees.joblib')
)

pred_et_tidal = et_tidal.predict(X_tidal_test_ohe)

In [11]:
# ============================================================
# 6-5. CatBoost 45% + ExtraTrees 55% 앙상블 예측
# ============================================================

pred_ensemble_tidal = pd.Series(
    pred_cat_tidal * 0.45 + pred_et_tidal * 0.55,
    index=X_tidal_test.index
)

# 앙상블 예측 결과 검증
assert pred_ensemble_tidal.index.equals(X_tidal_test.index)
assert len(pred_ensemble_tidal) == len(tidal_test)
assert pred_ensemble_tidal.notna().all()

print(f'Tidal Test: {len(tidal_test):,}행')
print(f'앙상블 예측: {len(pred_ensemble_tidal):,}행')
print('Tidal 앙상블 예측 검증 완료')

Tidal Test: 3,573행
앙상블 예측: 3,573행
Tidal 앙상블 예측 검증 완료


## 7. Normal 데이터셋 모델 생성: K_Fold 교차검증, CatBoost

In [12]:
# ============================================================
# 7-1. 고정 Fold CSV 불러오기 및 Train 데이터 매칭
# ============================================================

# 고정 Fold 매핑 파일 불러오기
fold_match = pd.read_csv(
    PROJECT_ROOT / 'data' / '0_kf_train_full_match.csv'
)

# event_id 기준으로 Fold 번호 매칭
normal_train['fold_num'] = normal_train['event_id'].map(
    fold_match.set_index('event_id')['fold_num']  # 매핑규칙: 'event_id'로 조인, 'fold_num' 부여
)

# 매칭 검증
assert normal_train['fold_num'].notna().all()
assert normal_train['fold_num'].between(1, 5).all()

print('Normal Fold 매칭 완료')

Normal Fold 매칭 완료


In [13]:
# ============================================================
# 7-2. Normal CatBoost 5-Fold 학습 및 예측
# ============================================================

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error
import numpy as np

# 결과 저장용 변수
normal_oof_pred = pd.Series(
    index=X_normal_train.index,
    dtype=float
)

normal_test_preds = pd.DataFrame(
    index=X_normal_test.index
)

normal_fold_results = []

# Fold별 학습
for fold in range(1, 6):

    print(f'\n========== Normal Fold {fold} ==========')

    # 기존 Fold 번호로 분할
    train_mask = normal_train['fold_num'] != fold
    valid_mask = normal_train['fold_num'] == fold

    X_kf_val = X_normal_train.loc[valid_mask]

    y_kf_val = y_normal_train.loc[valid_mask]

    assert len(X_kf_val) > 0, f'Fold {fold}: 검증 데이터 없음'

    # CatBoost 최종 선정모델
    model = CatBoostRegressor()

    model.load_model(
        str(PROJECT_ROOT / 'model' / f'cat_{fold}fold_normal.cbm')
    )

    # Fold별 kf_val 예측
    val_pred = model.predict(X_kf_val)

    # Fold별 OOF 예측 및 RMSE
    normal_oof_pred.loc[X_kf_val.index] = val_pred
    fold_oof_rmse = root_mean_squared_error(y_kf_val, val_pred)

    # Fold별 Test 전체 예측
    test_pred = model.predict(X_normal_test)
    normal_test_preds[f'fold_{fold}'] = test_pred  # Fold별 예측결과가 하나의 컬럼으로 삽입
  
    # Fold 결과 저장
    normal_fold_results.append({
        'fold': fold,
        'train_count': train_mask.sum(),
        'valid_count': len(X_kf_val),
        'oof_rmse': fold_oof_rmse,
    })

    print(
        f'Fold {fold} | '
        f'OOF RMSE={fold_oof_rmse:.6f} | '
    )


========== Normal Fold 1 ==========
Fold 1 | OOF RMSE=0.233810 | 

========== Normal Fold 2 ==========
Fold 2 | OOF RMSE=0.165635 | 

========== Normal Fold 3 ==========
Fold 3 | OOF RMSE=0.260627 | 

========== Normal Fold 4 ==========
Fold 4 | OOF RMSE=0.210118 | 

========== Normal Fold 5 ==========
Fold 5 | OOF RMSE=0.207717 | 


In [14]:
# ============================================================
# 7-3. Normal CatBoost 학습 및 예측 결과 검증
# ============================================================

# OOF 예측 검증
assert normal_oof_pred.index.equals(y_normal_train.index)
assert normal_oof_pred.notna().all()

# Test 예측 검증
assert normal_test_preds.index.equals(X_normal_test.index)
assert normal_test_preds.shape == (len(X_normal_test), 5)
assert normal_test_preds.notna().all().all()

# 전체 OOF RMSE
normal_oof_rmse = root_mean_squared_error(
    y_normal_train, normal_oof_pred
)

print(f'Normal 전체 OOF RMSE: {normal_oof_rmse:.6f}')
print(f'Normal Test 예측: {normal_test_preds.shape}')
print('Normal CatBoost 검증 완료')

Normal 전체 OOF RMSE: 0.217860
Normal Test 예측: (182690, 5)
Normal CatBoost 검증 완료


In [15]:
# ============================================================
# 7-4. Normal CatBoost 5-Fold 개별모델 앙상블 예측
# ============================================================

pred_ensemble_normal = normal_test_preds.mean(axis=1)  # axis=1은 행별집계
# 결과물: 하나의 행에 fold별 예측값이 컬럼별로 배치 -> 행별집계 -> 행별 1개 mean값(인덱스 유지)

# 검증
assert pred_ensemble_normal.index.equals(X_normal_test.index)
assert pred_ensemble_normal.notna().all()

print(f'Normal Test: {len(normal_test):,}행')
print(f'앙상블 예측: {len(pred_ensemble_normal):,}행')
print('Normal 앙상블 예측 검증 완료')

Normal Test: 182,690행
앙상블 예측: 182,690행
Normal 앙상블 예측 검증 완료


## 8. Tidal / Normal 예측 결과 통합 및 제출파일 생성

In [16]:
# ============================================================
# 8. Tidal / Normal 예측 결과 통합 및 제출파일 생성
# ============================================================

# 예측 결과 통합
pred_test = pd.concat([
    pred_ensemble_tidal,
    pred_ensemble_normal
])

# 제출용 데이터프레임 생성
final_result = pred_test.rename('pred_maxrise_6h').reset_index()

# 검증
assert final_result.columns.tolist() == ['row_id', 'pred_maxrise_6h']
assert len(final_result) == len(test)
assert final_result['row_id'].is_unique
assert set(final_result['row_id']) == set(test.index)
assert final_result['pred_maxrise_6h'].notna().all()

print('제출 데이터 생성 및 검증 완료')
print(final_result.shape)

제출 데이터 생성 및 검증 완료
(186263, 2)


In [17]:
final_result.to_csv(
    PROJECT_ROOT / 'data' / 'final_result_infer.csv',
    index=False,
    encoding='utf-8'
)